In [ ]:
import json, subprocess
r = subprocess.run(
    ["/mnt/hdd/mghan/so_difficulty_measure/rust-code-analysis-cli",
     "-m", "-p", "test.py", "-O", "json"],
    capture_output=True, text=True
)
print(json.dumps(json.loads(r.stdout), indent=2))

In [1]:
from lib.code_complexity.cyclomatic_complexity import call_cyclomatic_complexity, check_code

In [3]:
check_code('/mnt/hdd/mghan/so_difficulty_measure/result/code_complexity/cyclomatic_complexity/run_id_20000/data/src/1016271_76789405.py'
           , 'python')

True

In [4]:
check_code('/mnt/hdd/mghan/so_difficulty_measure/result/code_complexity/cyclomatic_complexity/run_id_20000/data/src/858021_67652254.py'
           , 'python')

False

In [1]:
import lib.code_complexity.parser_loader as ps
from setting_for_sdm.constants import CONSTANTS

In [2]:
CONSTANTS.src_extend.get('python')

'py'

In [5]:
CONSTANTS.src_extend.get('python') in CONSTANTS.lizard_supported


True

In [ ]:
lizard_lang

In [ ]:
from lib.code_complexity.cognitive_complexity_for_groovy import calculate_file

In [ ]:
from lib.code_complexity.cognitive_complexity_for_groovy import calculate_file

In [ ]:
calculate_file('/mnt/hdd/mghan/so_difficulty_measure/result/code_complexity/run_id_2018/data/src/4930_75956528.groovy')


In [ ]:
from lib.code_complexity.cognitive_complexity_for_java import calculate_file

In [ ]:
calculate_file('/mnt/hdd/mghan/so_difficulty_measure/result/code_complexity/run_id_1002/data/src/299700_75209707.java')

In [ ]:
from lib.code_complexity.cognitive_complexity import check_code

In [ ]:
check_code('/mnt/hdd/mghan/so_difficulty_measure/result/code_complexity/run_id_1002/data/src/299700_75209707.java', 'java')

In [ ]:
from lib.code_complexity.cognitive_complexity_for_kotlin import calculate_source

code = """private fun setupIndicators() {
        val indicators = arrayOfNulls<ImageView>(introSliderAdapter.itemCount)
        for (i in indicators.indices) {
            indicators[i] = ImageView(applicationContext)
        }
    }"""

r = calculate_source(code)
print(f"funcs={len(r)}, complexity={sum(x['complexity'] for x in r)}")

In [ ]:
from lib.code_complexity.cognitive_complexity_for_kotlin import create_parser

p = create_parser()
tree = p.parse(b"""private fun setupIndicators() {
    for (i in indicators.indices) {
        print(i)
    }
}""")

# 1. 어떤 파서를 쓰는지
print(type(p))

# 2. AST 확인
for ch in tree.root_node.children:
    print(f"root child: {ch.type}")
    for ch2 in ch.children:
        print(f"  {ch2.type}: {ch2.text.decode()[:30]}")
        if ch2.type == "function_body":
            for ch3 in ch2.children:
                print(f"    {ch3.type}")
                if ch3.type == "block":
                    for ch4 in ch3.children:
                        print(f"      {ch4.type}: {ch4.text.decode()[:30]}")

In [ ]:
import lib.code_complexity.parser_loader as ps
from lib.code_complexity.cognitive_complexity import check_code

CALC_FUNC = ps.CALC_FUNC
CALC_PARSER = ps.CALC_PARSER

# 0이 나오는 그 .groovy 파일의 실제 경로
file_path = '/mnt/hdd/mghan/so_difficulty_measure/result/code_complexity/run_id_2018/data/src/4930_75956528.groovy'
lang = "groovy"

# 1. 파일을 직접 읽어서 내용 확인
with open(file_path, 'r', encoding='utf-8') as f:
    content = f.read()
print(f"=== 파일 내용 (처음 500자) ===")
print(repr(content[:500]))
print(f"파일 크기: {len(content)} 문자")

# 2. parse 직접 시도
parser = CALC_PARSER[lang]()
parser.timeout_micros = 5_000_000
try:
    tree = parser.parse(bytes(content, "utf-8"))
    print(f"\n=== parse 결과 ===")
    print(f"has_error: {tree.root_node.has_error}")
    print(f"root type: {tree.root_node.type}")
    print(f"children count: {len(tree.root_node.children)}")
    print(f"top-level child types: {[c.type for c in tree.root_node.children[:10]]}")
except ValueError as e:
    print(f"parse failed: {e}")

# 3. check_code 결과
print(f"\n=== check_code ===")
print(f"check_code returns: {check_code(file_path, lang)}")

# 4. calculate_file 강제 호출 (check_code 우회)
print(f"\n=== calculate_file 강제 호출 ===")
results = CALC_FUNC[lang](file_path)
print(f"type: {type(results).__name__}")
print(f"length: {len(results) if hasattr(results, '__len__') else 'N/A'}")
print(f"results: {results}")

# 5. 함수 검출 직접 확인
def find_funcs(n, types, found=None):
    if found is None:
        found = []
    if n.type in types:
        name_node = n.child_by_field_name("name")
        name = content[name_node.start_byte:name_node.end_byte] if name_node else "<?>"
        found.append((n.type, name))
    for c in n.children:
        find_funcs(c, types, found)
    return found

print(f"\n=== AST에서 함수 직접 검색 ===")
funcs = find_funcs(tree.root_node, {
    'function_definition', 'function_declaration', 
    'method_declaration', 'constructor_declaration'
})
print(f"발견된 함수: {len(funcs)}")
for t, name in funcs[:20]:
    print(f"  {t}: {name}")

In [ ]:
import lib.code_complexity.parser_loader as ps

# 1. parser 객체 직접 확인
parser = ps.CALC_PARSER["groovy"]()
print(f"parser type: {type(parser)}")
print(f"parser module: {type(parser).__module__}")

# 2. language 객체 확인
if hasattr(parser, 'language'):
    lang = parser.language
    print(f"language: {lang}")
    print(f"language type: {type(lang)}")
    if hasattr(lang, 'name'):
        print(f"language name: {lang.name}")

# 3. parse 시도
code = b'void verifyOrder(List<String> Order){\n  for(String check:Order){\n    assert check\n  }\n}\n'
tree = parser.parse(code)
print(f"\nroot type: {tree.root_node.type}")
print(f"first child: {tree.root_node.children[0].type if tree.root_node.children else None}")

# 4. parser_loader.py에서 groovy 로딩 부분 확인
import inspect
src = inspect.getsource(ps)
# groovy 관련 줄 출력
for i, line in enumerate(src.split('\n'), 1):
    if 'groovy' in line.lower():
        print(f"  parser_loader line {i}: {line}")

In [ ]:
import lib.code_complexity.parser_loader as ps

# 1. CALC_PARSER 딕셔너리 키들
print("CALC_PARSER keys:", list(ps.CALC_PARSER.keys()))

# 2. groovy parser 객체
parser = ps.CALC_PARSER["groovy"]()
print(f"\nparser: {parser}")
print(f"parser class: {type(parser).__name__}")
print(f"parser module: {type(parser).__module__}")

# 3. language 정보
if hasattr(parser, 'language'):
    lang_obj = parser.language
    print(f"\nlanguage object: {lang_obj}")
    print(f"language type: {type(lang_obj)}")
    if hasattr(lang_obj, 'name'):
        print(f"language name: {lang_obj.name}")

# 4. 같은 코드를 직접 만든 groovy parser와 비교
try:
    import tree_sitter_groovy
    from tree_sitter import Parser, Language
    p2 = Parser(Language(tree_sitter_groovy.language()))
    code = b'void f(){for(int i:list){x()}}'
    
    t1 = parser.parse(code)
    t2 = p2.parse(code)
    
    print(f"\n=== parser_loader's parser ===")
    print(f"  root: {t1.root_node.type}")
    print(f"  child[0]: {t1.root_node.children[0].type if t1.root_node.children else None}")
    
    print(f"\n=== fresh tree_sitter_groovy parser ===")
    print(f"  root: {t2.root_node.type}")
    print(f"  child[0]: {t2.root_node.children[0].type if t2.root_node.children else None}")
    
    if t1.root_node.type != t2.root_node.type:
        print("\n>>> 두 parser가 다른 트리를 만듭니다! parser_loader가 잘못된 parser를 로드 중입니다.")
    else:
        print("\n>>> 두 parser가 동일한 트리를 만듭니다.")
except ImportError:
    print("\ntree_sitter_groovy not installed - try: pip install tree-sitter-groovy")

In [ ]:
pip install tree-sitter-groovy

In [ ]:
import tree_sitter_groovy
from tree_sitter import Parser, Language
p = Parser(Language(tree_sitter_groovy.language()))

code = b'void verifyOrder(List<String> Order){\n  for(String check:Order){\n    assert check\n  }\n}\n'
tree = p.parse(code)
print(f"root: {tree.root_node.type}")
print(f"child[0]: {tree.root_node.children[0].type}")
# 기대 결과:
#   root: program
#   child[0]: method_declaration

In [ ]:
import lib.code_complexity.parser_loader as ps

# 각 언어에 대해 간단한 valid 코드로 테스트
SAMPLES = {
    'java': ('class C { void f() { if (x) y(); } }', 'program', 'class_declaration'),
    'python': ('def f():\n    if x:\n        y()', 'module', 'function_definition'),
    'javascript': ('function f() { if (x) y(); }', 'program', 'function_declaration'),
    'typescript': ('function f(): void { if (x) y(); }', 'program', 'function_declaration'),
    'c': ('void f() { if (x) y(); }', 'translation_unit', 'function_definition'),
    'c++': ('void f() { if (x) y(); }', 'translation_unit', 'function_definition'),
    'c#': ('class C { void F() { if (x) Y(); } }', 'compilation_unit', 'class_declaration'),
    'go': ('package p\nfunc f() { if x { y() } }', 'source_file', 'package_clause'),
    'kotlin': ('fun f() { if (x) y() }', 'source_file', 'function_declaration'),
    'scala': ('object O { def f(): Unit = if (x) y() }', 'compilation_unit', 'object_definition'),
    'swift': ('func f() { if x { y() } }', 'source_file', 'function_declaration'),
    'rust': ('fn f() { if x { y(); } }', 'source_file', 'function_item'),
    'ruby': ('def f\n  if x then y end\nend', 'program', 'method'),
    'php': ('<?php function f() { if ($x) y(); }', 'program', 'php_tag'),
    'lua': ('function f() if x then y() end end', 'chunk', 'function_declaration'),
    'r': ('f <- function() { if (x) y() }', 'program', 'binary_operator'),
    'dart': ('void f() { if (x) y(); }', 'program', 'function_signature'),
    'groovy': ('void f() { if (x) y(); }', 'program', 'method_declaration'),
    'objective-c': ('void f() { if (x) y(); }', 'translation_unit', 'function_definition'),
    'matlab': ('function f()\n  if x\n    y()\n  end\nend', 'source_file', 'function_definition'),
    'julia': ('function f()\n  if x\n    y()\n  end\nend', 'source_file', 'function_definition'),
    'haskell': ('f x = if x then 1 else 0', 'haskell', 'function'),
    'perl': ('sub f { if ($x) { y(); } }', 'source_file', 'subroutine_declaration_statement'),
    'fortran': ('subroutine f\n  if (x) call y\nend subroutine', 'translation_unit', 'subroutine'),
}

print(f"{'lang':15} {'root':25} {'first_child':30} status")
print("-" * 80)
for lang in ['python', 'java', 'javascript', 'typescript', 'c', 'c++', 'c#',
             'go', 'kotlin', 'scala', 'swift', 'rust', 'ruby', 'php', 'lua',
             'r', 'dart', 'groovy', 'objective-c', 'matlab', 'julia',
             'haskell', 'perl', 'fortran', 'vb.net', 'delphi', 'f#',
             'solidity', 'prolog']:
    if lang not in SAMPLES:
        # Just check parse works
        try:
            parser = ps.CALC_PARSER[lang]()
            tree = parser.parse(b'x = 1')
            print(f"{lang:15} {tree.root_node.type:25} {'(no sample)':30} ?")
        except Exception as e:
            print(f"{lang:15} ERROR: {type(e).__name__}: {str(e)[:40]}")
        continue
    
    code, exp_root, exp_child = SAMPLES[lang]
    try:
        parser = ps.CALC_PARSER[lang]()
        tree = parser.parse(bytes(code, 'utf-8'))
        root = tree.root_node.type
        first = tree.root_node.children[0].type if tree.root_node.children else '(none)'
        
        # Find any function-like node anywhere in tree
        def find_func(n):
            if 'function' in n.type or 'method' in n.type or 'def' in n.type or 'subroutine' in n.type:
                return n.type
            for c in n.children:
                r = find_func(c)
                if r:
                    return r
            return None
        
        func = find_func(tree.root_node) or '(no func)'
        
        # Status: OK if function found and root looks reasonable
        status = "OK" if func != '(no func)' else "FAIL: no function"
        print(f"{lang:15} {root:25} {first:30} {status}  func={func}")
    except Exception as e:
        print(f"{lang:15} ERROR: {type(e).__name__}: {str(e)[:40]}")

In [ ]:
import lib.code_complexity.parser_loader as ps

EXTRA_SAMPLES = {
    'scala': 'object O { def f(x: Int): Int = if (x > 0) 1 else 0 }',
    'fortran': 'subroutine foo\n  integer :: i\n  do i = 1, 10\n    print *, i\n  end do\nend subroutine',
    'vb.net': 'Module M\nSub F()\n  If x Then\n    Y()\n  End If\nEnd Sub\nEnd Module',
    'delphi': 'procedure Foo;\nbegin\n  if x then y;\nend;',
    'f#': 'let f x = if x > 0 then 1 else 0',
    'solidity': 'pragma solidity ^0.8.0;\ncontract C { function f() public { if (true) {} } }',
    'prolog': 'foo(X) :- X > 0, write(X).',
    'assembly': 'mov eax, 1\njmp label\nlabel:',
    'typescript': 'function f(x: number): number { if (x > 0) return 1; return 0; }',
    'groovy': 'class C { void f() { if (x) y() } }',
    'perl': 'sub f { my $x = shift; if ($x) { return 1; } return 0; }',
}

print(f"{'lang':15} {'root':20} {'first':25} {'has_error':10} {'func':25}")
print("-" * 100)
for lang, code in EXTRA_SAMPLES.items():
    try:
        parser = ps.CALC_PARSER[lang]()
        try:
            parser.timeout_micros = 5_000_000
        except: pass
        tree = parser.parse(bytes(code, 'utf-8'))
        root = tree.root_node.type
        first = tree.root_node.children[0].type if tree.root_node.children else '(none)'
        err = tree.root_node.has_error
        
        # Try all common function-like node names
        def find_funcs(n, found=None):
            if found is None: found = set()
            if any(kw in n.type.lower() for kw in 
                   ['function', 'method', 'procedure', 'subroutine', 'predicate', 'definition']):
                found.add(n.type)
            for c in n.children:
                find_funcs(c, found)
            return found
        
        funcs = find_funcs(tree.root_node) or {'(none)'}
        print(f"{lang:15} {root:20} {first:25} {str(err):10} {','.join(funcs)[:25]}")
    except Exception as e:
        print(f"{lang:15} ERROR: {type(e).__name__}: {str(e)[:40]}")

In [ ]:
import tree_sitter_groovy
from tree_sitter import Parser, Language
p = Parser(Language(tree_sitter_groovy.language()))
tree = p.parse(b'class C { void f() { if (x) y() } }')
print(f"groovy OK: root={tree.root_node.type}, child={tree.root_node.children[0].type}")
# 기대: program / class_declaration

import tree_sitter_typescript
p = Parser(Language(tree_sitter_typescript.language_typescript()))
tree = p.parse(b'function f(x: number): number { return x; }')
print(f"typescript OK: root={tree.root_node.type}")
# 기대: program

In [ ]:
pip install tree-sitter-fortran

In [ ]:
import lib.code_complexity.parser_loader as ps
import tempfile, os

CALC_FUNC = ps.CALC_FUNC

# 각 언어의 myMethod-equivalent (모두 expected complexity = 6)
TESTS = {
    'java': ('class C { void f(boolean a, boolean b) { if (a) { for (int i=0;i<10;i++) { while (b) { body(); } } } } }', '.java', 6),
    'python': ('def f(a, b):\n    if a:\n        for i in range(10):\n            while b:\n                body()', '.py', 6),
    'javascript': ('function f(a, b) { if (a) { for (let i=0;i<10;i++) { while (b) { body(); } } } }', '.js', 6),
    'typescript': ('function f(a: boolean, b: boolean) { if (a) { for (let i=0;i<10;i++) { while (b) { body(); } } } }', '.ts', 6),
    'c': ('void f(int a, int b) { if (a) { for (int i=0;i<10;i++) { while (b) { body(); } } } }', '.c', 6),
    'c++': ('void f(int a, int b) { if (a) { for (int i=0;i<10;i++) { while (b) { body(); } } } }', '.cpp', 6),
    'c#': ('class C { void f(bool a, bool b) { if (a) { for (int i=0;i<10;i++) { while (b) { body(); } } } } }', '.cs', 6),
    'go': ('package p\nfunc f(a, b bool) { if a { for i := 0; i < 10; i++ { for b { body() } } } }', '.go', 6),
    'kotlin': ('fun f(a: Boolean, b: Boolean) { if (a) { for (i in 0..10) { while (b) { body() } } } }', '.kt', 6),
    'scala': ('object O { def f(a: Boolean, b: Boolean): Unit = { if (a) { for (i <- 0 until 10) { while (b) { body() } } } } }', '.scala', 6),
    'swift': ('func f(a: Bool, b: Bool) { if a { for i in 0..<10 { while b { body() } } } }', '.swift', 6),
    'rust': ('fn f(a: bool, b: bool) { if a { for i in 0..10 { while b { body(); } } } }', '.rs', 6),
    'ruby': ('def f(a, b)\n  if a\n    for i in 0..10\n      while b\n        body\n      end\n    end\n  end\nend', '.rb', 6),
    'php': ('<?php function f($a, $b) { if ($a) { for ($i=0;$i<10;$i++) { while ($b) { body(); } } } }', '.php', 6),
    'lua': ('function f(a, b)\n  if a then\n    for i = 1, 10 do\n      while b do\n        body()\n      end\n    end\n  end\nend', '.lua', 6),
    'r': ('f <- function(a, b) {\n  if (a) {\n    for (i in 1:10) {\n      while (b) {\n        body()\n      }\n    }\n  }\n}', '.r', 6),
    'dart': ('void f(bool a, bool b) { if (a) { for (var i=0;i<10;i++) { while (b) { body(); } } } }', '.dart', 6),
    'groovy': ('class C { void f(boolean a, boolean b) { if (a) { for (int i=0;i<10;i++) { while (b) { body() } } } } }', '.groovy', 6),
    'objective-c': ('@implementation Foo\n- (void)fa:(BOOL)a b:(BOOL)b {\n    if (a) { for (int i=0;i<10;i++) { while (b) { body(); } } }\n}\n@end', '.m', 6),
    'matlab': ('function f(a, b)\n    if a\n        for i = 1:10\n            while b\n                body();\n            end\n        end\n    end\nend', '.m', 6),
    'julia': ('function f(a, b)\n    if a\n        for i in 1:10\n            while b\n                body()\n            end\n        end\n    end\nend', '.jl', 6),
    'haskell': ('f a b = if a then map g [1..10] else []\n  where g i = if b then body i else 0', '.hs', None),  # Haskell pattern is different
    'perl': ('sub f { my ($a, $b) = @_; if ($a) { for (my $i=0; $i<10; $i++) { while ($b) { body(); } } } }', '.pl', 6),
}

print(f"{'lang':15} {'expected':10} {'got':10} {'status'}")
print("-" * 50)
for lang, (code, ext, expected) in TESTS.items():
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix=ext, delete=False) as f:
            f.write(code)
            path = f.name
        try:
            results = CALC_FUNC[lang](path)
            total = sum(r['complexity'] for r in results)
            n = len(results)
            if expected is None:
                status = "?"
            else:
                status = "OK" if total == expected else f"WRONG ({n} funcs)"
            print(f"{lang:15} {str(expected):<10} {total:<10} {status}")
        finally:
            os.unlink(path)
    except Exception as e:
        print(f"{lang:15} ERROR: {type(e).__name__}: {str(e)[:40]}")

In [ ]:
import lib.code_complexity.parser_loader as ps

# 1. 사용자 환경의 kotlin parser가 만드는 트리 확인
parser = ps.CALC_PARSER['kotlin']()
code = b'''fun f(a: Boolean, b: Boolean) {
    if (a) {
        for (i in 0..10) {
            while (b) {
                body()
            }
        }
    }
}'''
tree = parser.parse(code)
print(f"root: {tree.root_node.type}")
print(f"has_error: {tree.root_node.has_error}")

def show(n, depth=0, max_depth=8):
    if depth > max_depth: return
    print('  '*depth + n.type)
    for c in n.children:
        show(c, depth+1, max_depth)
show(tree.root_node, max_depth=6)

# 2. calculate_file 강제 호출하면 어떻게 나오는지
import tempfile, os
with tempfile.NamedTemporaryFile(mode='w', suffix='.kt', delete=False) as f:
    f.write(code.decode())
    path = f.name

try:
    results = ps.CALC_FUNC['kotlin'](path)
    print(f"\nresults: {results}")
    for r in results:
        print(f"  {r['function']}: {r['complexity']}")
        for d in r.get('details', []):
            print(f"    {d}")
finally:
    os.unlink(path)

# 3. calculator 파일 위치와 last modified time 확인
import lib.code_complexity.cognitive_complexity_for_kotlin as kotlin_mod
import os
path = kotlin_mod.__file__
print(f"\nCalculator file: {path}")
print(f"Last modified: {os.path.getmtime(path)}")
import datetime
print(f"             = {datetime.datetime.fromtimestamp(os.path.getmtime(path))}")


In [ ]:
# 사용자 환경에서 실행
import lib.code_complexity.cognitive_complexity_for_kotlin as kotlin_mod

# 1. 모듈이 import한 것 확인
parser = kotlin_mod.create_parser()
print(f"parser: {parser}")
print(f"language: {parser.language}")
if hasattr(parser.language, 'name'):
    print(f"language name: {parser.language.name}")

# 2. 간단한 kotlin 코드 파싱해서 root 노드 확인
code = b'fun f() { if (x) y() }'
tree = parser.parse(code)
print(f"\nroot: {tree.root_node.type}")
print(f"first child: {tree.root_node.children[0].type if tree.root_node.children else None}")

# 3. language_pack이 kotlin을 가지고 있는지
import tree_sitter_language_pack as tslp
print(f"\ntslp has_language('kotlin'): {tslp.has_language('kotlin')}")

# 4. 언어 이름이 진짜 kotlin인지 확인
try:
    lang = tslp.get_language('kotlin')
    print(f"tslp get_language('kotlin'): {lang}")
    if hasattr(lang, 'name'):
        print(f"  name: {lang.name}")
except Exception as e:
    print(f"tslp get_language ERROR: {e}")

# 5. 개별 tree_sitter_kotlin 패키지가 설치되어 있는지
import importlib.util
spec = importlib.util.find_spec('tree_sitter_kotlin')
print(f"\ntree_sitter_kotlin installed: {spec is not None}")
if spec:
    import tree_sitter_kotlin
    from tree_sitter import Parser, Language
    p2 = Parser(Language(tree_sitter_kotlin.language()))
    t2 = p2.parse(code)
    print(f"  direct parse root: {t2.root_node.type}")
    print(f"  first child: {t2.root_node.children[0].type if t2.root_node.children else None}")

In [ ]:
import lib.code_complexity.cognitive_complexity_for_kotlin as k

code = '''fun f(a: Boolean, b: Boolean) {
    if (a) {
        for (i in 0..10) {
            while (b) {
                body()
            }
        }
    }
}'''

r = k.calculate_source(code)
print(f"complexity: {sum(x['complexity'] for x in r)} (expected 6)")

In [ ]:
import lib.code_complexity.cognitive_complexity_for_kotlin as k
import inspect

# 1. 파일 위치
print(f"File: {inspect.getsourcefile(k)}")

# 2. create_parser 소스 출력 (try 순서가 뒤집혔는지 확인)
print(inspect.getsource(k.create_parser))

# 3. 어떤 parser가 실제로 로드되는지
parser = k.create_parser()
code = b'fun f() { if (x) y() }'
tree = parser.parse(code)
print(f"\nroot: {tree.root_node.type}")
print(f"first child: {tree.root_node.children[0].type if tree.root_node.children else None}")
print(f"has_error: {tree.root_node.has_error}")

# 4. tree_sitter_kotlin 직접 import 가능한지
try:
    import tree_sitter_kotlin
    print(f"\ntree_sitter_kotlin: OK")
    from tree_sitter import Parser, Language
    p2 = Parser(Language(tree_sitter_kotlin.language()))
    t2 = p2.parse(code)
    print(f"  direct root: {t2.root_node.type}")
    print(f"  direct first: {t2.root_node.children[0].type}")
except ImportError as e:
    print(f"\ntree_sitter_kotlin: NOT INSTALLED ({e})")

In [ ]:
import lib.code_complexity.cognitive_complexity_for_kotlin as k

code = '''fun f(a: Boolean, b: Boolean) {
    if (a) {
        for (i in 0..10) {
            while (b) {
                body()
            }
        }
    }
}'''

# 1. parse + 전체 트리 출력 (max_depth 충분히)
parser = k.create_parser()
tree = parser.parse(bytes(code, 'utf-8'))

def show(n, depth=0, max_depth=20):
    if depth > max_depth: return
    print('  '*depth + n.type)
    for c in n.children:
        show(c, depth+1, max_depth)

print("=== FULL TREE ===")
show(tree.root_node, max_depth=20)

# 2. calculate 결과 자세히
print("\n=== RESULTS ===")
r = k.calculate_source(code)
for x in r:
    print(f"function: {x['function']}")
    print(f"complexity: {x['complexity']}")
    print(f"start_line: {x['start_line']}")
    print(f"end_line: {x['end_line']}")
    print("details:")
    for d in x.get('details', []):
        print(f"  {d}")

In [ ]:
import lib.code_complexity.cognitive_complexity_for_kotlin as k

code = '''fun f(a: Boolean, b: Boolean) {
    if (a) {
        for (i in 0..10) {
            while (b) {
                body()
            }
        }
    }
}'''

parser = k.create_parser()
tree = parser.parse(bytes(code, 'utf-8'))

# if_expression 노드 찾아서 그 children 자세히 보기
def find_node(n, target_type):
    if n.type == target_type:
        return n
    for c in n.children:
        r = find_node(c, target_type)
        if r:
            return r
    return None

if_node = find_node(tree.root_node, 'if_expression')
print(f"if_expression found: {if_node is not None}")
if if_node:
    print(f"\nif_expression children (with field names):")
    for i, c in enumerate(if_node.children):
        # field name 확인
        field = if_node.field_name_for_child(i) if hasattr(if_node, 'field_name_for_child') else None
        print(f"  [{i}] type='{c.type}' field='{field}' text={c.text[:30] if c.text else b''!r}")
    
    # condition / consequence / alternative 시도
    print(f"\nfield-based access:")
    print(f"  condition: {if_node.child_by_field_name('condition')}")
    print(f"  consequence: {if_node.child_by_field_name('consequence')}")
    print(f"  alternative: {if_node.child_by_field_name('alternative')}")
    print(f"  body: {if_node.child_by_field_name('body')}")
    print(f"  then: {if_node.child_by_field_name('then')}")

# for_statement 또는 다른 이름의 loop 노드 찾기
print("\n=== Loop-like nodes in tree ===")
def find_all(n, found=None):
    if found is None: found = []
    if any(kw in n.type for kw in ['for', 'while', 'loop']):
        found.append((n.type, n.start_point[0]+1))
    for c in n.children:
        find_all(c, found)
    return found
for t, line in find_all(tree.root_node):
    print(f"  line {line}: {t}")

# control_structure_body 같은 wrapper가 있는지
print("\n=== All node types in tree ===")
def collect_types(n, types=None):
    if types is None: types = set()
    types.add(n.type)
    for c in n.children:
        collect_types(c, types)
    return types
all_types = collect_types(tree.root_node)
print(sorted(all_types))

In [ ]:
import lib.code_complexity.cognitive_complexity_for_kotlin as k

# Force reload (Python이 캐싱했을 수 있음)
import importlib
importlib.reload(k)

code = '''fun f(a: Boolean, b: Boolean) {
    if (a) {
        for (i in 0..10) {
            while (b) {
                body()
            }
        }
    }
}'''

r = k.calculate_source(code)
total = sum(x['complexity'] for x in r)
print(f"complexity: {total} (expected 6)")
for x in r:
    print(f"  {x['function']}: {x['complexity']}")
    for d in x.get('details', []):
        print(f"    {d}")

In [ ]:
import lib.code_complexity.cognitive_complexity_for_kotlin as k

code = 'fun f(a: Boolean, b: Boolean) { }'
parser = k.create_parser()
tree = parser.parse(bytes(code, 'utf-8'))

# function_declaration 노드 찾기
def find(n, t):
    if n.type == t: return n
    for c in n.children:
        r = find(c, t)
        if r: return r
    return None

fd = find(tree.root_node, 'function_declaration')
print(f"function_declaration children:")
for i, c in enumerate(fd.children):
    field = fd.field_name_for_child(i) if hasattr(fd, 'field_name_for_child') else None
    text = c.text[:30] if c.text else b''
    print(f"  [{i}] type='{c.type}' field='{field}' text={text!r}")

print(f"\nfield-based:")
print(f"  name: {fd.child_by_field_name('name')}")
print(f"  identifier: {fd.child_by_field_name('identifier')}")

In [ ]:
import lib.code_complexity.cognitive_complexity_for_prolog as p

# 다양한 prolog 패턴
samples = {
    'fact':        'foo(1).',
    'simple_rule': 'foo(X) :- bar(X).',
    'conjunction': 'foo(X) :- X > 0, bar(X), baz(X).',
    'disjunction': 'foo(X) :- bar(X) ; baz(X).',
    'if_then_else':'foo(X, Y) :- (X > 0 -> Y = 1 ; Y = 0).',
    'nested_ite':  'foo(X, Y) :- (X > 10 -> Y = big ; X > 0 -> Y = med ; Y = sml).',
    'multi_clause':'foo([]).\nfoo([_|T]) :- foo(T).',
    'findall':     'foo(L, S) :- findall(X, member(X, L), S).',
    'forall':      'foo(L) :- forall(member(X, L), X > 0).',
    'cut':         'foo(X) :- bar(X), !, baz(X).',
    'negation':    'foo(X) :- \\+ bar(X).',
    'catch':       'foo(X) :- catch(bar(X), Error, handle(Error)).',
}

parser = p.create_parser()

# 1. 모든 노드 타입 수집
all_types = set()
for code in samples.values():
    tree = parser.parse(code.encode())
    def collect(n):
        all_types.add(n.type)
        for c in n.children:
            collect(c)
    collect(tree.root_node)

print("=== All node types ===")
for t in sorted(all_types):
    print(f"  {t}")

# 2. 각 샘플의 트리 구조
def show(n, depth=0, max_depth=12, lines=None):
    if lines is None: lines = []
    if depth > max_depth:
        return lines
    text = ''
    if not n.children:
        # leaf
        try:
            text = f' {n.text.decode()[:20]!r}'
        except:
            pass
    lines.append('  '*depth + n.type + text)
    for c in n.children:
        show(c, depth+1, max_depth, lines)
    return lines

for label, code in samples.items():
    print(f"\n=== {label}: {code!r} ===")
    tree = parser.parse(code.encode())
    print(f"has_error: {tree.root_node.has_error}")
    lines = show(tree.root_node)
    for line in lines:
        print(line)

In [ ]:
LANGS_TO_CHECK = ['prolog', 'fortran', 'f#', 'delphi']

import lib.code_complexity.parser_loader as ps

for lang in LANGS_TO_CHECK:
    print(f"\n{'='*60}\n{lang.upper()}\n{'='*60}")
    
    parser = ps.CALC_PARSER[lang]()
    
    samples = {
        'prolog':  'foo(X, Y) :- (X > 0 -> Y = 1 ; Y = 0).',
        'fortran': 'subroutine f\n  integer :: i\n  do i = 1, 10\n    if (i > 5) print *, i\n  end do\nend subroutine',
        'f#':  'let f x = if x > 0 then for i in 1..10 do printfn "%d" i',
        'delphi':  'procedure Foo;\nvar i: Integer;\nbegin\n  for i := 1 to 10 do\n    if x then y;\nend;',
    }
    
    code = samples[lang]
    tree = parser.parse(code.encode())
    print(f"has_error: {tree.root_node.has_error}")
    
    all_types = set()
    def collect(n):
        all_types.add(n.type)
        for c in n.children:
            collect(c)
    collect(tree.root_node)
    print(f"node types: {sorted(all_types)}")
    
    def show(n, depth=0, max_depth=8):
        if depth > max_depth: return
        print('  '*depth + n.type)
        for c in n.children:
            show(c, depth+1, max_depth)
    show(tree.root_node, max_depth=6)

In [ ]:
import os

base = "/home/mghan/sopjt/git/so_difficulty_measure/lib/code_complexity"
for so in ['prolog.so', 'fsharp.so']:
    paths = [
        f"{base}/build/{so}",
        f"{base}/{so}",
    ]
    for p in paths:
        if os.path.exists(p):
            print(f"FOUND: {p}")
            break
    else:
        print(f"NOT FOUND: {so}")

In [ ]:
import importlib
import lib.code_complexity.cognitive_complexity_for_prolog as p
importlib.reload(p)

# parser 정보
parser = p.create_parser()
print(f"language: {parser.language}")
if hasattr(parser.language, 'name'):
    print(f"name: {parser.language.name}")

# 간단한 prolog 코드 파싱 - 이전과 비교
code = b'foo(X, Y) :- (X > 0 -> Y = 1 ; Y = 0).'
tree = parser.parse(code)
print(f"\nroot: {tree.root_node.type}")
print(f"first child: {tree.root_node.children[0].type if tree.root_node.children else None}")
print(f"has_error: {tree.root_node.has_error}")

# foxyseta grammar는 'clause_term'을 직접 children으로 가져야 함
# 이전 grammar는 'compound_term'을 가졌음

In [ ]:
import importlib
import lib.code_complexity.cognitive_complexity_for_prolog as p
importlib.reload(p)

tests = {
    'fact':         ('foo(1).', 0),
    'simple_rule':  ('foo(X) :- bar(X).', 0),
    'conjunction':  ('foo(X) :- bar(X), baz(X), qux(X).', 0),
    'if_then_else': ('foo(X, Y) :- (X > 0 -> Y = 1 ; Y = 0).', 2),
    'multi_clause': ('foo([]).\nfoo([_|T]) :- foo(T).', 0),
    'findall':      ('foo(L) :- findall(X, member(X, L), L2), bar(L2).', 0),
    'nested_ite':   ('foo(X, Y) :- (X > 10 -> Y = big ; X > 0 -> Y = med ; Y = sml).', 4),
}

print(f"{'pattern':15} {'expected':10} {'got':10} {'status':10}")
print("-" * 55)
for label, (code, expected) in tests.items():
    r = p.calculate_source(code)
    total = sum(x['complexity'] for x in r)
    n_funcs = len(r)
    status = "OK" if total == expected else f"WRONG"
    print(f"{label:15} {expected:<10} {total:<10} {status} ({n_funcs} funcs)")

In [ ]:
import importlib
import lib.code_complexity.cognitive_complexity_for_fsharp as fs
importlib.reload(fs)

# parser 정보 먼저
parser = fs.create_parser()
print(f"language: {parser.language}")

code = b'let f x = if x > 0 then 1 else 0'
tree = parser.parse(code)
print(f"root: {tree.root_node.type}")
print(f"first child: {tree.root_node.children[0].type if tree.root_node.children else None}")
print(f"has_error: {tree.root_node.has_error}")

# 결과 확인
r = fs.calculate_source(code.decode())
print(f"\nresults: {len(r)}")
for x in r:
    print(f"  {x['function']}: {x['complexity']}")

In [ ]:
import importlib
import lib.code_complexity.cognitive_complexity_for_fsharp as fs
importlib.reload(fs)

# 마지막 개행 있는 코드
codes = [
    ('한 줄 + \\n',           'let f x = if x > 0 then 1 else 0\n'),
    ('multi-line + \\n',       'let f x =\n    if x > 0 then 1 else 0\n'),
    ('module + \\n',           'module M\nlet f x = if x > 0 then 1 else 0\n'),
    ('larger',                 '''module M

let f x =
    if x > 0 then
        for i in 1..10 do
            printfn "%d" i
        1
    else
        0
'''),
]

for label, code in codes:
    parser = fs.create_parser()
    tree = parser.parse(bytes(code, 'utf-8'))
    r = fs.calculate_source(code)
    total = sum(x['complexity'] for x in r)
    print(f"{label:25} has_error={tree.root_node.has_error}  complexity={total}  ({len(r)} funcs)")

In [ ]:
import importlib
import lib.code_complexity.cognitive_complexity_for_fsharp as fs
importlib.reload(fs)

code = '''module M

let f x =
    if x > 0 then
        for i in 1..10 do
            printfn "%d" i
        1
    else
        0
'''

parser = fs.create_parser()
tree = parser.parse(bytes(code, 'utf-8'))
print(f"root: {tree.root_node.type}")
print(f"has_error: {tree.root_node.has_error}")

def show(n, depth=0, max_depth=12):
    if depth > max_depth: return
    print('  '*depth + n.type)
    for c in n.children:
        show(c, depth+1, max_depth)
show(tree.root_node, max_depth=10)

# 모든 노드 타입
def collect(n, types=None):
    if types is None: types = set()
    types.add(n.type)
    for c in n.children:
        collect(c, types)
    return types

print(f"\nAll node types: {sorted(collect(tree.root_node))}")

In [ ]:
import importlib
import lib.code_complexity.cognitive_complexity_for_fsharp as fs
importlib.reload(fs)

# 이전 테스트
code = '''module M

let f x =
    if x > 0 then
        for i in 1..10 do
            printfn "%d" i
        1
    else
        0
'''

r = fs.calculate_source(code)
print(f"results: {len(r)}")
for x in r:
    print(f"  {x['function']}: {x['complexity']}")
    for d in x.get('details', []):
        print(f"    {d}")

In [ ]:
import importlib
import lib.code_complexity.cognitive_complexity_for_delphi as d
importlib.reload(d)

code = '''procedure Foo;
var i: Integer;
begin
  for i := 1 to 10 do
  begin
    if x then y;
  end;
end;'''

# parser 정보
parser = d.create_parser()
print(f"language: {parser.language}")
if hasattr(parser.language, 'name'):
    print(f"name: {parser.language.name}")

tree = parser.parse(bytes(code, 'utf-8'))
print(f"\nroot: {tree.root_node.type}")
print(f"has_error: {tree.root_node.has_error}")

# calculate
r = d.calculate_source(code)
print(f"\nresults: {len(r)}")
for x in r:
    print(f"  {x['function']}: {x['complexity']}")
    for det in x.get('details', []):
        print(f"    {det}")

In [ ]:
import importlib
import lib.code_complexity.cognitive_complexity_for_delphi as d
importlib.reload(d)

code = '''procedure Foo;
var i: Integer;
begin
  for i := 1 to 10 do
  begin
    if x then y;
  end;
end;'''

r = d.calculate_source(code)
print(f"results: {len(r)}")
for x in r:
    print(f"  {x['function']}: {x['complexity']}")
    for det in x.get('details', []):
        print(f"    {det}")

In [ ]:
import importlib
import lib.code_complexity.cognitive_complexity_for_delphi as d
importlib.reload(d)

code = '''procedure Foo;
var i: Integer;
begin
  for i := 1 to 10 do
  begin
    if x then y;
  end;
end;'''

parser = d.create_parser()
tree = parser.parse(bytes(code, 'utf-8'))

# 1. 전체 트리 출력
def show(n, depth=0, max_depth=15):
    if depth > max_depth: return
    text = ''
    if not n.children:
        try: text = f' {n.text.decode()[:20]!r}'
        except: pass
    print('  '*depth + n.type + text)
    for c in n.children:
        show(c, depth+1, max_depth)

print("=== TREE ===")
show(tree.root_node)

# 2. root의 직접 children
print(f"\n=== root direct children ===")
print(f"root type: {tree.root_node.type}")
for i, c in enumerate(tree.root_node.children):
    print(f"  [{i}] {c.type}")

# 3. defProc이 어디 있는지 찾기
def find_paths(n, target, path=""):
    found = []
    if n.type == target:
        found.append(path)
    for c in n.children:
        found.extend(find_paths(c, target, f"{path}/{c.type}"))
    return found

print(f"\n=== defProc locations ===")
for p in find_paths(tree.root_node, 'defProc'):
    print(f"  {p}")

In [ ]:
import importlib
import lib.code_complexity.cognitive_complexity_for_delphi as d
importlib.reload(d)

code = '''procedure Foo;
var i: Integer;
begin
  for i := 1 to 10 do
  begin
    if x then y;
  end;
end;'''

r = d.calculate_source(code)
print(f"results: {len(r)}")
for x in r:
    print(f"  {x['function']}: {x['complexity']}")
    for det in x.get('details', []):
        print(f"    {det}")